# 3D Point Cloud Segmentation
Segment exactly one sphere (yellow) and multiple planes (distinct colors) from `.pcd` to `.ply` using RANSAC from scratch.

**Constraints:** NumPy and OpenCV only. No 3D geometry libraries.

## 1. Custom PCD Parser
- Reads ASCII PCD files directly.
- Extracts spatial $(x, y, z)$ coordinates only; filters invalid sensor depth ($z \le 0$).

In [1]:
import os
import sys
import math
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt

def read_pcd(file_path, filter_zeros=True):
    """Reads ASCII PCD file returning (N, 3) spatial coordinates."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
        
    header = {}
    data_started = False
    points_list = []
    
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
                
            if not data_started:
                tokens = line.split()
                keyword = tokens[0].upper()
                if keyword == 'DATA':
                    header['DATA'] = tokens[1] if len(tokens) > 1 else 'ascii'
                    data_started = True
                else:
                    header[keyword] = tokens[1:]
            else:
                parts = line.split()
                if len(parts) >= 3:
                    points_list.append([float(parts[0]), float(parts[1]), float(parts[2])])
                    
    points = np.array(points_list, dtype=np.float64)
    raw_count = len(points)
    
    if filter_zeros and len(points) > 0:
        valid_mask = (points[:, 2] > 0.0) & (np.linalg.norm(points, axis=1) > 1e-3)
        points = points[valid_mask]
        
    return points, header, raw_count

In [2]:
sample_file = 'Frame4994.pcd'
if not os.path.exists(sample_file):
    sample_file = os.path.join('FIRST_ASSIGNMENT', 'Frame4994.pcd')

points, header, raw_count = read_pcd(sample_file, filter_zeros=True)
print(f"Parsed {os.path.basename(sample_file)}: {len(points):,} valid points (filtered {raw_count - len(points):,} sensor zeros)")
print(f"X: [{points[:, 0].min():.2f}, {points[:, 0].max():.2f}] mm | "
      f"Y: [{points[:, 1].min():.2f}, {points[:, 1].max():.2f}] mm | "
      f"Z: [{points[:, 2].min():.2f}, {points[:, 2].max():.2f}] mm")

Parsed Frame4994.pcd: 215,659 valid points (filtered 1,429 sensor zeros)
X: [-962.14, 711.52] mm | Y: [-667.50, 741.24] mm | Z: [259.44, 1779.51] mm
